# 📐 Floor Plan Reader: Dataset Preparation Pipeline
### Stage 1: CubiCasa5k + Synthetic Floor Plan Data Mixture

This notebook prepares the multimodal instruction-tuning dataset for fine-tuning **Qwen3-VL 8B** on Google Colab.

**Workflow:**
1. Mount Google Drive to persist preprocessed datasets.
2. Download & parse **CubiCasa5k** floor plans from Hugging Face into normalized `[ymin, xmin, ymax, xmax]` bounding boxes (0-1000 scale).
3. Ingest custom synthetic floor plans from your **floor plan generator tool** (supporting JPG, PNG, and SVG vector drawings).
4. Blend both datasets into a joint mixture (**Strategy 1**: 70% CubiCasa5k, 30% Synthetic) to prevent catastrophic forgetting.
5. Export `train.jsonl` and `val.jsonl` formatted for Hugging Face `SFTTrainer`.

In [ ]:
# Step 1: Mount Google Drive (Recommended for Colab persistence)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/floorplan_reader_project'
except Exception:
    import os
    PROJECT_DIR = './data/colab_run'
    print("Running locally or outside Google Colab.")

import os
os.makedirs(f"{PROJECT_DIR}/processed", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/images", exist_ok=True)
print(f"Project directory ready at: {PROJECT_DIR}")

In [ ]:
# Step 2: Install required libraries
!pip install -q --upgrade pip
!pip install -q datasets pillow pydantic cairosvg svglib reportlab matplotlib

In [ ]:
# Step 3: Clone or install floorplan_reader repository
!git clone https://github.com/your-username/floorPlanReader.git /content/floorPlanReader 2>/dev/null || true
import sys
if '/content/floorPlanReader' not in sys.path:
    sys.path.append('/content/floorPlanReader')

# If running inside local workspace
import os
if os.path.exists('./floorplan_reader') and '.' not in sys.path:
    sys.path.append('.')

from floorplan_reader.dataset.cubicasa_parser import CubiCasaParser
from floorplan_reader.dataset.synthetic_adapter import (
    SyntheticFloorPlanAdapter,
    generate_mock_synthetic_dataset,
)
from floorplan_reader.dataset.dataset_mixer import DatasetMixer
print("FloorPlanReader dataset tools loaded successfully!")

### Step 4: Stream and Ingest CubiCasa5k from Hugging Face
We load a curated split of the CubiCasa5k dataset and convert annotations to the normalized 0-1000 scale.

In [ ]:
from datasets import load_dataset
from tqdm.auto import tqdm
from pathlib import Path

print("Loading CubiCasa5k dataset from Hugging Face Hub (streaming mode)... ")
cubi_parser = CubiCasaParser()
cubicasa_samples = []

CUBICASA_SAMPLE_LIMIT = 500  # Adjust as needed (e.g. 500-1500 for Colab T4 fast fine-tuning)
cubi_images_dir = Path(f"{PROJECT_DIR}/images/cubicasa")
cubi_images_dir.mkdir(parents=True, exist_ok=True)

try:
    # Load pre-packaged COCO-style CubiCasa5k
    hf_dataset = load_dataset("phungpx/cubicasa5k-coco", split="train", streaming=True)
    count = 0
    for sample in tqdm(hf_dataset, total=CUBICASA_SAMPLE_LIMIT, desc="Processing CubiCasa5k"):
        if count >= CUBICASA_SAMPLE_LIMIT:
            break
        img = sample["image"]
        annotations = sample.get("annotations", [])
        w, h = img.size
        sid = f"cubi_{count:04d}"
        entry = cubi_parser.parse_coco_sample(
            image_path_or_pil=img,
            annotations=annotations,
            img_width=w,
            img_height=h,
            sample_id=sid,
            output_image_dir=cubi_images_dir,
        )
        cubicasa_samples.append(entry)
        count += 1
    print(f"Successfully parsed {len(cubicasa_samples)} CubiCasa5k samples.")
except Exception as e:
    print(f"Note: HuggingFace streaming error ({e}). Generating mock CubiCasa samples for demonstration.")
    # Fallback mock generator
    for i in range(10):
        from PIL import Image
        mock_img = Image.new('RGB', (800, 600), (255, 255, 255))
        rec = cubi_parser.parse_coco_sample(
            image_path_or_pil=mock_img,
            annotations=[
                {"bbox": [50, 50, 300, 200], "category_name": "Bedroom"},
                {"bbox": [400, 50, 350, 450], "category_name": "Living Room"},
                {"bbox": [380, 100, 20, 80], "category_name": "Door"}
            ],
            img_width=800,
            img_height=600,
            sample_id=f"cubi_mock_{i:03d}",
            output_image_dir=cubi_images_dir,
        )
        cubicasa_samples.append(rec)

### Step 5: Ingest Your Floor Plan Generator Dataset
Provide the directory path to your synthetic floor plans (paired images/SVGs and JSON annotations).
If you haven't generated your dataset yet, the adapter automatically generates mock synthetic plans for testing.

In [ ]:
# Set your synthetic dataset directory here (or leave None to generate mock plans)
MY_SYNTHETIC_DATA_DIR = f"{PROJECT_DIR}/my_synthetic_data"

synth_adapter = SyntheticFloorPlanAdapter()
synth_images_dir = Path(f"{PROJECT_DIR}/images/synthetic")
synth_images_dir.mkdir(parents=True, exist_ok=True)

if os.path.exists(MY_SYNTHETIC_DATA_DIR) and len(os.listdir(MY_SYNTHETIC_DATA_DIR)) > 0:
    print(f"Ingesting custom synthetic floor plans from {MY_SYNTHETIC_DATA_DIR}...")
    synthetic_samples = synth_adapter.ingest_paired_directory(
        data_dir=MY_SYNTHETIC_DATA_DIR,
        output_images_dir=synth_images_dir,
    )
else:
    print("Custom dataset folder not found or empty. Generating 50 synthetic vector floor plans for testing...")
    mock_source_dir = Path(f"{PROJECT_DIR}/mock_synthetic_source")
    generate_mock_synthetic_dataset(mock_source_dir, num_samples=50, format="svg")
    synthetic_samples = synth_adapter.ingest_paired_directory(
        data_dir=mock_source_dir,
        output_images_dir=synth_images_dir,
    )

print(f"Successfully ingested {len(synthetic_samples)} synthetic floor plans.")

### Step 6: Blend into Joint Training Mixture (Strategy 1)
Combine 70% CubiCasa5k + 30% Synthetic data to retain real-world blueprint generalization while specializing on your generator's layouts.

In [ ]:
processed_dir = Path(f"{PROJECT_DIR}/processed")
mixer = DatasetMixer(cubicasa_ratio=0.7, val_split_ratio=0.1, seed=42)

summary = mixer.build_and_export(
    cubicasa_samples=cubicasa_samples,
    synthetic_samples=synthetic_samples,
    output_dir=processed_dir,
)

print("\n--- Dataset Summary ---")
print(f"Total Samples: {summary['total_samples']}")
print(f"Training Set:  {summary['train_samples']} samples -> {summary['train_file']}")
print(f"Val Set:       {summary['val_samples']} samples -> {summary['val_file']}")
print("Ready for LoRA training in 02_train_qwen_vl_lora_colab.ipynb!")